In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
BASE_ENV = Path().resolve().parent

In [ ]:
# load column names
df_columns = pd.read_csv(BASE_ENV / 'data/raw/NUSW-NB15_features.csv', encoding='windows-1252')

In [ ]:
frames = []
for i in range(1, 5):
    temp = pd.read_csv(BASE_ENV / f'data/raw/UNSW-NB15_{i}.csv', header=None)
    temp.columns = df_columns['Name'].to_list()
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
columns_to_drop = ['srcip',
                    'dstip',
                    'sport',
                    'dsport',
                    'Ltime',
                    'Label',
                    'ct_dst_src_ltm',
                    'proto_arp',
                    'ct_src_dport_ltm',
                    'ct_srv_src',
                    'service_dns',
                    'service_pop3',
                    'dtcpb',
                    'state_REQ',
                    'proto_ospf',
                    'service_http',
                    'proto_tcp',
                    'ct_ftp_cmd',
                    'service_smtp',
                    'proto_sctp',
                    'state_URH',
                    'proto_icmp',
                    'trans_depth',
                    'ct_flw_http_mthd',
                    'is_ftp_login',
                    'proto_swipe',
                    'dwin',
                    'proto_sun-nd',
                    'proto_mobile',
                    'proto_unas',
                    'state_RST',
                    'service_ssl',
                    'service_ftp',
                    'proto_rsvp',
                    'service_dhcp',
                    'res_bdy_len',
                    'service_ssh',
                    'state_TXD',
                    'state_MAS',
                    'state_ECO',
                    'proto_rvd',
                    'proto_sep',
                    'service_snmp',
                    'proto_trunk-2',
                    'proto_gre',
                    'proto_secure-vmtp',
                    'proto_aes-sp3-d',
                    'proto_tp++',
                    'proto_any',
                    'proto_ptp',
                    'proto_xtp',
                    'proto_ipv6',
                    'proto_compaq-peer',
                    'proto_ipx-n-ip',
                    'proto_br-sat-mon',
                    'proto_aris',
                    'proto_pim',
                    'proto_iso-ip',
                    'proto_bbn-rcc',
                    'proto_xnet',
                    'proto_stp',
                    'proto_nvp',
                    'state_CLO',
                    'proto_sprite-rpc',
                    'proto_sat-expak',
                    'proto_rtp',
                    'proto_ipv6-route',
                    'proto_dgp',
                    'proto_wb-mon',
                    'proto_iso-tp4',
                    'proto_cphb',
                    'proto_trunk-1',
                    'proto_wsn',
                    'proto_vmtp',
                    'proto_leaf-1',
                    'proto_snp',
                    'proto_xns-idp',
                    'proto_prm',
                    'proto_idpr-cmtp',
                    'proto_argus',
                    'proto_cpnx',
                    'proto_ddx',
                    'proto_sccopmce',
                    'proto_cbt',
                    'proto_netblt',
                    'proto_fire',
                    'proto_egp',
                    'proto_ddp',
                    'proto_emcon',
                    'proto_eigrp',
                    'proto_chaos',
                    'proto_crtp',
                    'proto_cftp',
                    'proto_a/n',
                    'proto_ax.25',
                    'proto_bna',
                    'proto_crudp',
                    'proto_dcn',
                    'proto_ifmp',
                    'proto_igp',
                    'proto_rdp',
                    'proto_qnx',
                    'proto_pvp',
                    'proto_pup',
                    'proto_pri-enc',
                    'proto_pgm',
                    'proto_pipe',
                    'proto_pnni',
                    'proto_nsfnet-igp',
                    'proto_merit-inp',
                    'proto_mhrp',
                    'proto_mfe-nsp',
                    'proto_micp',
                    'proto_mtp',
                    'proto_mux',
                    'proto_narp',
                    'proto_isis',
                    'proto_larp',
                    'proto_l2tp',
                    'proto_kryptolan',
                    'proto_leaf-2',
                    'proto_il',
                    'proto_ip',
                    'proto_ipcomp',
                    'proto_ipnip',
                    'proto_iplt',
                    'proto_ipip',
                    'proto_ippc',
                    'proto_ipv6-no',
                    'proto_ipv6-frag',
                    'proto_ipv6-opts',
                    'proto_irtp',
                    'proto_encap',
                    'proto_etherip',
                    'proto_ggp',
                    'proto_ib',
                    'proto_i-nlsp',
                    'proto_iatp',
                    'proto_gmtp',
                    'proto_hmp',
                    'proto_fc',
                    'proto_idpr',
                    'proto_idrp',
                    'proto_ipcv',
                    'proto_smp',
                    'proto_skip',
                    'proto_sat-mon',
                    'proto_scps',
                    'proto_vines',
                    'proto_uti',
                    'proto_ttp',
                    'proto_tlsp',
                    'proto_srp',
                    'proto_st2',
                    'proto_tcf',
                    'proto_sps',
                    'proto_sm',
                    'proto_sdrp',
                    'proto_visa',
                    'proto_vrrp',
                    'proto_zero',
                    'proto_wb-expak',
                    'service_radius',
                    'service_irc'
                ]

In [ ]:
df = df.replace('-', np.nan)


In [ ]:
df['service'] = df['service'].fillna('Unknown')

In [ ]:
df['attack_cat'] = df['attack_cat'].replace(np.nan, 'Benign')
df['attack_cat'] = df['attack_cat'].str.strip()
df['attack_cat'] = df['attack_cat'].replace('Backdoors', 'Backdoor')

In [ ]:
df['attack_cat'] = df['attack_cat'].str.strip()

In [ ]:
df_normal = df[df['attack_cat'] == 'Benign']
df_attacks = df[df['attack_cat'] != 'Benign']

total_attacks = len(df_attacks)
df_normal_undersampled = df_normal.sample(n=total_attacks * 2, random_state=42)
df_balanced = pd.concat([df_normal_undersampled, df_attacks])

In [ ]:
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
categorical_features = ['proto', 'state', 'service']
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

In [ ]:
df = df.drop(columns_to_drop, axis=1)

In [ ]:
merge_map = {
    'Analysis': 'Rare_Attack',
    'Backdoor': 'Rare_Attack',
}
df['attack_cat'] = df['attack_cat'].replace(merge_map)

In [ ]:
def temporal_split(data, train_ratio=0.70, val_ratio=0.10):
    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return (
        data.iloc[:train_end],
        data.iloc[train_end:val_end],
        data.iloc[val_end:]
    )

In [ ]:
# Separate after balancing
benign_df = df[df['attack_cat'] == 'Benign'].sort_values('Stime')
attack_df = df[df['attack_cat'] != 'Benign'].sort_values('Stime')

benign_train, benign_val, benign_test = temporal_split(benign_df)
attack_train, attack_val, attack_test = temporal_split(attack_df)

train_df = pd.concat([benign_train, attack_train]).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([benign_val, attack_val]).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([benign_test, attack_test]).sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
for split in [train_df, val_df, test_df]:
    split.drop(columns='Stime', errors='ignore', inplace=True)

X_train, y_train = train_df.drop(columns=['attack_cat']), train_df['attack_cat']
X_val,   y_val   = val_df.drop(columns=['attack_cat']),   val_df['attack_cat']
X_test,  y_test  = test_df.drop(columns=['attack_cat']),  test_df['attack_cat']

In [ ]:
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")
print(f"Total: {len(X_train) + len(X_val) + len(X_test)}")

print("\n--- Train Attack Distribution ---")
print(y_train.value_counts())

print("\n--- Val Attack Distribution ---")
print(y_val.value_counts())

print("\n--- Test Attack Distribution ---")
print(y_test.value_counts())

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)


In [ ]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)
y_val_encoded = encoder.transform(y_val)

In [ ]:
ART = BASE_ENV / 'artifacts'

In [ ]:
# ART = BASE_ENV / "artifacts"

# train
np.save(ART / "datasets/train/X.npy", X_train_scaled)
np.save(ART / "datasets/train/y.npy", y_train_encoded)

# test
np.save(ART / "datasets/test/X.npy", X_test_scaled)
np.save(ART / "datasets/test/y.npy", y_test_encoded)

# val
np.save(ART / "datasets/val/X.npy", X_val_scaled)
np.save(ART / "datasets/val/y.npy", y_val_encoded)

# scaler
joblib.dump(scaler, ART / "preprocessors/scaler.joblib")

# encoder
joblib.dump(encoder, ART / "preprocessors/encoder.joblib")

In [ ]:
df.columns